# Brain Tumor MRI Classification - EfficientNetB0 Fine-Tuning

## Objective

Improve the baseline EfficientNetB0 model through fine-tuning.

The baseline model uses a frozen EfficientNetB0 backbone.
In this experiment, selected upper layers of the backbone will be unfrozen
to allow the model to adapt its learned features to the brain MRI dataset.

## Baseline

Validation Accuracy: 86.16%

The main classification errors observed in the baseline model include:

- Glioma → Meningioma
- Glioma → Pituitary
- Meningioma → Pituitary

The goal of fine-tuning is to improve class-specific feature extraction,
particularly for glioma and meningioma.

## Experiment Strategy

- Load the baseline EfficientNetB0 model
- Freeze most pretrained layers
- Unfreeze selected upper layers
- Keep Batch Normalization layers frozen
- Use a lower learning rate
- Train with early stopping
- Compare performance with the baseline model

## Import Library

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import load_model

print("TensorFlow version:", tf.__version__)

2026-08-30 12:18:09.606341: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/macbookpro2019/MaziyaCode/brain-tumor-classification-ai/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow version: 2.14.0


## Project Configuration

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR = DATA_DIR / "Testing"

MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

BASELINE_MODEL_PATH = MODEL_DIR / "efficientnetb0_baseline.h5"
FINETUNED_MODEL_PATH = MODEL_DIR / "efficientnetb0_finetuned.h5"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("Baseline model:", BASELINE_MODEL_PATH)
print("Model exists:", BASELINE_MODEL_PATH.exists())

Baseline model: ../models/efficientnetb0_baseline.h5
Model exists: True


## Load Dataset

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_dataset.class_names

print("Classes:", class_names)

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


Optimasi Pipeline

In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    AUTOTUNE
)

## Load Baseline Model

In [5]:
baseline_model = tf.keras.models.load_model(
    BASELINE_MODEL_PATH
)

baseline_model.summary()

Model: "brain_tumor_efficientnetb0"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 224, 224, 3)]     0         
                                                                 
 data_augmentation (Sequent  (None, 224, 224, 3)       0         
 ial)                                                            
                                                                 
 efficientnetb0 (Functional  (None, 7, 7, 1280)        4049571   
 )                                                               
                                                                 
 global_average_pooling (Gl  (None, 1280)              0         
 obalAveragePooling2D)                                           
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                        

## 5. Fine-Tuning Configuration

The baseline model uses EfficientNetB0 as a frozen feature extractor.

In this experiment, the last 20 layers of the EfficientNetB0 backbone
will be unfrozen to allow the model to adapt its high-level features
to brain MRI images.

Batch Normalization layers remain frozen to improve training stability.

In [6]:
# ==========================================
# Fine-Tuning Configuration
# ==========================================

backbone = baseline_model.get_layer("efficientnetb0")

# Aktifkan backbone
backbone.trainable = True

# Freeze semua layer terlebih dahulu
for layer in backbone.layers:
    layer.trainable = False

# Unfreeze 20 layer terakhir
for layer in backbone.layers[-20:]:
    layer.trainable = True

# Batch Normalization tetap frozen
for layer in backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Backbone trainable:", backbone.trainable)

print("\nTrainable layers:")
for layer in backbone.layers:
    if layer.trainable:
        print("-", layer.name)

Backbone trainable: True

Trainable layers:
- block6d_project_conv
- block6d_drop
- block6d_add
- block7a_expand_conv
- block7a_expand_activation
- block7a_dwconv
- block7a_activation
- block7a_se_squeeze
- block7a_se_reshape
- block7a_se_reduce
- block7a_se_expand
- block7a_se_excite
- block7a_project_conv
- top_conv
- top_activation


In [7]:
# ==========================================
# Recompile Model for Fine-Tuning
# ==========================================

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("Model recompiled successfully.")

Model recompiled successfully.


In [8]:
trainable_params = np.sum([
    np.prod(variable.shape)
    for variable in baseline_model.trainable_variables
])

non_trainable_params = np.sum([
    np.prod(variable.shape)
    for variable in baseline_model.non_trainable_variables
])

total_params = trainable_params + non_trainable_params

print("Trainable parameters:", trainable_params)
print("Non-trainable parameters:", non_trainable_params)
print("Total parameters:", total_params)

Trainable parameters: 1347892
Non-trainable parameters: 2706803.0
Total parameters: 4054695.0


## 6. Fine-Tuning Training

The fine-tuned model is trained using a lower learning rate to allow
the pretrained EfficientNetB0 weights to adapt gradually to the MRI
dataset.

Early stopping and learning-rate reduction are used to reduce the
risk of overfitting.

In [9]:
# ==========================================
# Fine-Tuning Callbacks
# ==========================================

FINETUNED_MODEL_PATH = MODEL_DIR / "efficientnetb0_finetuned.h5"

callbacks_finetuning = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FINETUNED_MODEL_PATH),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

print("Fine-tuning callbacks configured.")
print("Model will be saved to:", FINETUNED_MODEL_PATH)

Fine-tuning callbacks configured.
Model will be saved to: ../models/efficientnetb0_finetuned.h5


Training

In [10]:
EPOCHS_FINETUNING = 10

fine_tuning_history = baseline_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_FINETUNING,
    callbacks=callbacks_finetuning
)

Epoch 1/10
140/140 [==============================] - ETA: 0s - loss: 0.2650 - accuracy: 0.9025
Epoch 1: val_accuracy improved from -inf to 0.99286, saving model to ../models/efficientnetb0_finetuned.h5


/Users/macbookpro2019/MaziyaCode/brain-tumor-classification-ai/.venv/lib/python3.9/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


140/140 [==============================] - 228s 2s/step - loss: 0.2650 - accuracy: 0.9025 - val_loss: 0.0285 - val_accuracy: 0.9929 - lr: 1.0000e-04
Epoch 2/10
140/140 [==============================] - ETA: 0s - loss: 0.2024 - accuracy: 0.9270
Epoch 2: val_accuracy improved from 0.99286 to 0.99732, saving model to ../models/efficientnetb0_finetuned.h5
140/140 [==============================] - 228s 2s/step - loss: 0.2024 - accuracy: 0.9270 - val_loss: 0.0077 - val_accuracy: 0.9973 - lr: 1.0000e-04
Epoch 3/10
140/140 [==============================] - ETA: 0s - loss: 0.1608 - accuracy: 0.9404
Epoch 3: val_accuracy did not improve from 0.99732
140/140 [==============================] - 242s 2s/step - loss: 0.1608 - accuracy: 0.9404 - val_loss: 0.0143 - val_accuracy: 0.9973 - lr: 1.0000e-04
Epoch 4/10
140/140 [==============================] - ETA: 0s - loss: 0.1254 - accuracy: 0.9547
Epoch 4: val_accuracy improved from 0.99732 to 0.99821, saving model to ../models/efficientnetb0_finetun

In [11]:
history_df = pd.DataFrame(fine_tuning_history.history)

history_df

,loss,accuracy,val_loss,val_accuracy,lr
0,0.265031,0.902455,0.028488,0.992857,0.00010
1,0.202355,0.927009,0.007695,0.997321,0.00010
2,0.160803,0.940402,0.014314,0.997321,0.00010
3,0.125392,0.954687,0.007833,0.998214,0.00010
4,0.101611,0.964286,0.009212,0.998214,0.00002
5,0.090435,0.969643,0.005678,0.999107,0.00002
6,0.086386,0.970982,0.004630,0.999107,0.00002
7,0.077290,0.974330,0.003314,1.000000,0.00002
8,0.079211,0.971875,0.004163,0.999107,0.00002
9,0.074915,0.973661,0.003474,0.999107,0.00002


In [12]:
best_epoch = history_df["val_accuracy"].idxmax()

print("Best Epoch:", best_epoch + 1)
print(
    "Best Validation Accuracy:",
    history_df.loc[best_epoch, "val_accuracy"]
)

Best Epoch: 8
Best Validation Accuracy: 1.0


In [13]:
print("Fine-tuned model saved:", FINETUNED_MODEL_PATH.exists())
print("Path:", FINETUNED_MODEL_PATH)

Fine-tuned model saved: True
Path: ../models/efficientnetb0_finetuned.h5
